# 06 - Avaliacao das colunas para os modelos

Esta etapa le `csvs/03_dataset_limpo_sem_corridas_encurtadas.csv`, saida do `03_Analise_Outliers.ipynb`, e classifica cada coluna quanto a utilidade para os modelos de machine learning. O arquivo e apenas lido: nenhuma linha ou coluna e alterada e nenhum dataset novo e salvo.

**Premissa.** O alvo e `rd_position_display_order`, a ordem oficial completa (1 = vencedor, nao classificados no fim), o mesmo de `04_Correlation.ipynb` e `07_Refining_Dataset.ipynb`. A previsao e feita **antes da largada**. Toda coluna que so e conhecida depois da largada e inutil para o modelo, ou pior, vaza o resultado.

As justificativas usam os resultados de `04_Correlation.ipynb` e a regra `leakage_risk` de `feature_engineering.py`. As celulas de cada secao conferem nos dados as afirmacoes que podem ser checadas diretamente.

In [1]:
import re

import pandas as pd
from IPython.display import display

from feature_engineering import leakage_risk

ALVO = 'rd_position_display_order'
df = pd.read_csv('csvs/03_dataset_limpo_sem_corridas_encurtadas.csv', low_memory=False)
print(f'Linhas: {len(df):,} | Colunas: {df.shape[1]} | Corridas: {df.race_id.nunique()}')

Linhas: 8,462 | Colunas: 137 | Corridas: 403


## 1. Classificacao das colunas

Cada coluna recebe uma unica categoria. As categorias 1 a 8 sao inuteis na forma bruta; a 9 depende do modelo; a ultima reune o que sobra de util.

| Categoria | Motivo |
|---|---|
| 1. duplicata do alvo | Mesma informacao do alvo em outra forma. |
| 2. resultado da propria corrida | So existe depois da largada: vazamento direto. |
| 3. standing apos a corrida | Classificacao do campeonato ja atualizada com a corrida. |
| 4. resumo de carreira atual | Totais e melhores marcas de hoje, repetidos em todas as corridas: incluem o futuro. |
| 5. agregado da temporada | Listas da temporada inteira, sem indicar o que valia em cada etapa. |
| 6. identificador ou texto redundante | Nome que repete um id, texto que repete um numero, data bruta. |
| 7. medida redundante | Mesma medida de outra coluna mantida. |
| 8. tempo bruto ou sinal fraco | Mede a pista e nao o piloto, ou quase nao se relaciona com o alvo. |
| 9. constante dentro da corrida | Igual para todos os pilotos da corrida; nao ordena pilotos sozinha. |
| util pre-largada | Conhecida antes da largada e informativa. |

In [2]:
RESUMOS = re.compile(r'^(driver|constructor|engmfr|tyremfr)_(total|best)_')

categorias = {
    'alvo': [ALVO],
    '1. duplicata do alvo': ['rd_position_number', 'rd_position_text', 'rd_race_points'],
    '2. resultado da propria corrida': [
        'rd_race_laps', 'rd_race_positions_gained', 'rd_race_pit_stops', 'rd_race_fastest_lap',
        'rd_race_driver_of_the_day', 'rd_race_grand_slam', 'sess_fastest_lap_time_millis',
    ],
    '3. standing apos a corrida': (
        [c for c in df.columns if c.startswith(('dstand_', 'cstand_'))]
        + ['race_drivers_championship_decider', 'race_constructors_championship_decider']
    ),
    '4. resumo de carreira atual': (
        [c for c in df.columns if RESUMOS.match(c)]
        + ['circuit_total_races_held', 'gp_total_races_held']
    ),
    '5. agregado da temporada': [
        'ed_entrant_ids', 'ed_entrant_names', 'ed_rounds_text', 'ed_test_driver',
        'chassis_ids', 'chassis_names', 'engine_ids', 'engine_names',
    ],
    '6. identificador ou texto redundante': [
        'race_id', 'race_date', 'driver_date_of_birth',
        'driver_name', 'constructor_name', 'engmfr_name', 'tyremfr_name', 'circuit_name',
        'rd_race_grid_position_text', 'sess_grid_position_text', 'sess_quali_position_text',
    ],
    '7. medida redundante': [
        'rd_race_grid_position_number', 'rd_race_pole_position', 'sess_grid_time_millis',
        'layout_length', 'circuit_length', 'circuit_turns', 'race_laps', 'race_distance',
        'engine_capacities', 'engine_aspirations', 'engine_configurations', 'sess_quali_laps',
    ],
    '8. tempo bruto ou sinal fraco': [
        'sess_fp1_time_millis', 'sess_fp2_time_millis', 'sess_fp3_time_millis',
        'sess_quali_time_millis', 'sess_quali_q1_millis', 'sess_quali_q2_millis', 'sess_quali_q3_millis',
        'sess_fp1_laps', 'sess_fp2_laps', 'sess_fp3_laps', 'tyremfr_id',
    ],
    '9. constante dentro da corrida': [
        'race_year', 'race_round', 'race_qualifying_format', 'race_direction',
        'race_course_length', 'race_turns', 'circuit_id', 'circuit_type',
        'layout_id', 'layout_effective', 'gp_id',
    ],
    'util pre-largada': [
        'sess_grid_position_number', 'sess_quali_position_number',
        'sess_fp1_position_number', 'sess_fp2_position_number', 'sess_fp3_position_number',
        'driver_id', 'constructor_id', 'engmfr_id',
    ],
}

todas = pd.Series([coluna for colunas in categorias.values() for coluna in colunas])
assert not todas.duplicated().any(), todas[todas.duplicated()].tolist()
assert set(todas) == set(df.columns), (set(df.columns) - set(todas), set(todas) - set(df.columns))

classificacao = pd.Series(
    {coluna: categoria for categoria, colunas in categorias.items() for coluna in colunas},
    name='categoria',
)
risco = pd.Series({coluna: leakage_risk(coluna) for coluna in df.columns}, name='leakage_risk')

# Nenhuma coluna mantida pode estar marcada como vazamento por feature_engineering.py.
mantidas = categorias['util pre-largada'] + categorias['9. constante dentro da corrida']
assert risco[mantidas].isna().all(), risco[mantidas].dropna()

print(f'Colunas classificadas: {len(classificacao)} de {df.shape[1]}')
display(classificacao.value_counts().sort_index().rename('colunas').to_frame())

Colunas classificadas: 137 de 137


,colunas
categoria,
1. duplicata do alvo,3
2. resultado da propria corrida,7
3. standing apos a corrida,12
4. resumo de carreira atual,53
5. agregado da temporada,8
6. identificador ou texto redundante,11
7. medida redundante,12
8. tempo bruto ou sinal fraco,11
9. constante dentro da corrida,11


## 2. Duplicatas do alvo

| Coluna | Justificativa |
|---|---|
| `rd_position_number` | E a mesma ordem do alvo, restrita aos classificados (Spearman 1,000 no `04_Correlation.ipynb`). |
| `rd_position_text` | Repete a coluna acima, com DNF/DNS/DSQ/NC no lugar dos numeros. |
| `rd_race_points` | Deriva direto da posicao final e so existe para quem pontuou. |

In [3]:
classificados = df['rd_position_number'].notna()
igual_alvo = df.loc[classificados, ALVO].eq(df.loc[classificados, 'rd_position_number']).mean()
print(f'Alvo = rd_position_number nas linhas classificadas: {igual_alvo:.2%}')

pontuou = df['rd_race_points'].notna().map({True: 'com pontos', False: 'sem pontos'})
display(df.groupby(pontuou)[ALVO].agg(['count', 'min', 'max']))

Alvo = rd_position_number nas linhas classificadas: 100.00%


,count,min,max
rd_race_points,,,
com pontos,3800,1,10
sem pontos,4662,7,24


## 3. Resultado da propria corrida (vazamento)

| Coluna | Justificativa |
|---|---|
| `rd_race_laps` | Voltas completadas pelo piloto. Um abandono ja aparece aqui. |
| `rd_race_positions_gained` | Grid menos posicao final: contem o alvo. |
| `rd_race_pit_stops` | So e conhecido durante a corrida. |
| `rd_race_fastest_lap`, `sess_fastest_lap_time_millis` | Volta mais rapida da propria prova. |
| `rd_race_driver_of_the_day` | Votacao feita depois da corrida; so existe a partir de 2016. |
| `rd_race_grand_slam` | Exige vencer a corrida: contem o alvo. |

In [4]:
ganho_calculado = df['rd_race_grid_position_number'] - df['rd_position_number']
comparaveis = ganho_calculado.notna() & df['rd_race_positions_gained'].notna()
igual_ganho = df.loc[comparaveis, 'rd_race_positions_gained'].eq(ganho_calculado[comparaveis]).mean()
print(f'positions_gained = grid - posicao final: {igual_ganho:.2%} de {comparaveis.sum():,} linhas comparaveis')

grand_slam_sem_vitoria = (df['rd_race_grand_slam'].eq(1) & df[ALVO].ne(1)).sum()
print(f'Grand slams sem vitoria: {grand_slam_sem_vitoria}')

primeiro_dotd = df.loc[df['rd_race_driver_of_the_day'].notna(), 'race_year'].min()
print(f'Primeiro ano com piloto do dia: {primeiro_dotd}')

fracao_voltas = (df['rd_race_laps'] / df['race_laps']).rename('voltas_completadas / voltas_da_corrida')
situacao = classificados.map({True: 'classificado', False: 'nao classificado'})
display(fracao_voltas.groupby(situacao).describe()[['count', 'mean', '25%', '50%', '75%']].round(3))

positions_gained = grid - posicao final: 100.00% de 6,775 linhas comparaveis
Grand slams sem vitoria: 0
Primeiro ano com piloto do dia: 2016


,count,mean,25%,50%,75%
rd_position_number,,,,,
classificado,6898.0,0.990,0.983,1.000,1.000
nao classificado,1501.0,0.381,0.097,0.359,0.645


## 4. Standings apos a corrida (vazamento)

`dstand_*` e `cstand_*` sao a classificacao do campeonato **ja atualizada com a propria corrida**. Na primeira rodada da temporada, os pontos do campeonato sao os pontos daquela corrida. `race_drivers_championship_decider` e `race_constructors_championship_decider` dependem do resultado. `cstand_engine_manufacturer_id`, alem disso, repete `engmfr_id`.

In [5]:
primeira_rodada = df['race_round'].eq(1)
pontos_iguais = (
    df.loc[primeira_rodada, 'dstand_points'].fillna(0)
    .eq(df.loc[primeira_rodada, 'rd_race_points'].fillna(0))
)
print(f'Rodada 1: dstand_points = pontos da corrida em {pontos_iguais.sum()} de {len(pontos_iguais)} linhas')

com_motor = df['cstand_engine_manufacturer_id'].notna()
igual_motor = df.loc[com_motor, 'cstand_engine_manufacturer_id'].eq(df.loc[com_motor, 'engmfr_id']).mean()
print(f'cstand_engine_manufacturer_id = engmfr_id: {igual_motor:.2%}')

Rodada 1: dstand_points = pontos da corrida em 380 de 380 linhas
cstand_engine_manufacturer_id = engmfr_id: 100.00%


## 5. Resumos de carreira atuais (incluem o futuro)

`driver_*`, `constructor_*`, `engmfr_*` e `tyremfr_*` com `total` ou `best`, alem de `circuit_total_races_held` e `gp_total_races_held`. Se o valor for o mesmo em todas as corridas de um piloto ou equipe, ele e uma fotografia de hoje: em 2003 o modelo ja "saberia" o que o piloto conquistou depois. O substituto correto sao as features `driver_prior_*` e `constructor_prior_*` de `07_Refining_Dataset.ipynb`, calculadas com `shift(1)`.

In [6]:
linhas = []
for grupo in ['driver', 'constructor', 'engmfr', 'tyremfr']:
    colunas = [c for c in df.columns if RESUMOS.match(c) and c.startswith(grupo + '_')]
    valores_por_entidade = df.groupby(grupo + '_id')[colunas].nunique().max()
    linhas.append({
        'grupo': grupo,
        'colunas': len(colunas),
        'max_valores_por_entidade': valores_por_entidade.max(),
    })
display(pd.DataFrame(linhas))

exemplo = df.loc[df['driver_id'].eq('michael-schumacher')].sort_values('race_date')
display(exemplo[['race_date', 'driver_total_race_wins', 'driver_total_championship_wins']].head(3))

,grupo,colunas,max_valores_por_entidade
0,driver,14,1
1,constructor,15,1
2,engmfr,14,1
3,tyremfr,8,1


,race_date,driver_total_race_wins,driver_total_championship_wins
3,2003-03-09,91,7
25,2003-03-23,91,7
40,2003-04-20,91,7


## 6. Agregados da temporada (vazamento sutil)

| Coluna | Justificativa |
|---|---|
| `ed_rounds_text` | Etapas disputadas na temporada inteira. Revela substituicoes futuras ja na rodada 1. |
| `chassis_*`, `engine_ids`, `engine_names` | Listam tudo o que foi usado na temporada, inclusive pecas que so estrearam depois. |
| `ed_entrant_*` | Agregado da temporada; na pratica e a equipe com o nome do patrocinador. |
| `ed_test_driver` | Quase constante: quem larga raramente e piloto de testes. Tambem e um agregado da temporada, sem dizer o papel do piloto naquela etapa. |

In [7]:
primeira_corrida = df.loc[df['race_date'].eq(df['race_date'].min())]
display(primeira_corrida[['driver_name', 'ed_rounds_text', 'chassis_names', 'engine_names']])

multiplos_chassis = df['chassis_ids'].str.contains(',', na=False).mean()
multiplos_motores = df['engine_ids'].str.contains(',', na=False).mean()
print(f'Linhas com mais de um chassi listado: {multiplos_chassis:.1%}')
print(f'Linhas com mais de um motor listado: {multiplos_motores:.1%}')

equipe_temporada = df[['constructor_id', 'race_year']].drop_duplicates().shape[0]
print(f'Chassis distintos: {df.chassis_ids.nunique()} | Combinacoes equipe x temporada: {equipe_temporada}')
display(df['ed_test_driver'].value_counts(dropna=False).rename('linhas').to_frame())

,driver_name,ed_rounds_text,chassis_names,engine_names
0,David Coulthard,1-16,MP4-17D,"FO 110M,FO 110P"
1,Juan Pablo Montoya,1-16,FW25,P83
2,Kimi Räikkönen,1-16,MP4-17D,"FO 110M,FO 110P"
3,Michael Schumacher,1-16,"F2002B,F2003-GA","051B,052"
4,Jarno Trulli,1-16,"R23,R23B",RS23
5,Heinz-Harald Frentzen,1-16,C22,03A
6,Fernando Alonso,1-16,"R23,R23B",RS23
7,Ralf Schumacher,1-16,FW25,P83
8,Jacques Villeneuve,1-15,005,RA003E
9,Jenson Button,1-16,005,RA003E


Linhas com mais de um chassi listado: 5.9%
Linhas com mais de um motor listado: 1.5%
Chassis distintos: 253 | Combinacoes equipe x temporada: 253


,linhas
ed_test_driver,
0,8365
1,97


## 7. Identificadores e textos redundantes

- `race_id` e `race_date` servem para agrupar e dividir os dados no tempo, nao como feature.
- `driver_date_of_birth` so e util convertida em idade na data da corrida (`driver_age_years` em `07_Refining_Dataset.ipynb`).
- Nomes repetem o id correspondente.
- As colunas `*_position_text` repetem a coluna numerica, com codigos (PL, NC, EX, DSQ) no lugar dos nulos. Se os codigos forem uteis, uma flag binaria basta.

In [8]:
pares_nome = [
    ('driver_id', 'driver_name'), ('constructor_id', 'constructor_name'),
    ('engmfr_id', 'engmfr_name'), ('tyremfr_id', 'tyremfr_name'), ('circuit_id', 'circuit_name'),
    ('ed_entrant_ids', 'ed_entrant_names'), ('chassis_ids', 'chassis_names'), ('engine_ids', 'engine_names'),
]
display(pd.DataFrame([
    {
        'id': id_,
        'nome': nome,
        'max_nomes_por_id': df.groupby(id_)[nome].nunique().max(),
        'max_ids_por_nome': df.groupby(nome)[id_].nunique().max(),
    }
    for id_, nome in pares_nome
]))

pares_texto = [
    ('rd_race_grid_position_text', 'rd_race_grid_position_number'),
    ('sess_grid_position_text', 'sess_grid_position_number'),
    ('sess_quali_position_text', 'sess_quali_position_number'),
]
linhas = []
for texto, numero in pares_texto:
    convertido = pd.to_numeric(df[texto], errors='coerce')
    numerico = convertido.notna()
    codigos = df.loc[df[texto].notna() & ~numerico, texto].value_counts()
    linhas.append({
        'coluna_texto': texto,
        'igual_a_numerica': convertido[numerico].eq(df.loc[numerico, numero]).mean(),
        'codigos': ', '.join(f'{codigo} ({n})' for codigo, n in codigos.items()),
    })
display(pd.DataFrame(linhas))

,id,nome,max_nomes_por_id,max_ids_por_nome
0,driver_id,driver_name,1,1
1,constructor_id,constructor_name,1,1
2,engmfr_id,engmfr_name,1,1
3,tyremfr_id,tyremfr_name,1,1
4,circuit_id,circuit_name,1,1
5,ed_entrant_ids,ed_entrant_names,1,1
6,chassis_ids,chassis_names,1,2
7,engine_ids,engine_names,1,2


,coluna_texto,igual_a_numerica,codigos
0,rd_race_grid_position_text,1.0,PL (162)
1,sess_grid_position_text,1.0,PL (118)
2,sess_quali_position_text,1.0,"NC (82), EX (16), DSQ (8)"


## 8. Medidas redundantes

- **Grid:** `rd_race_grid_position_number` e `sess_grid_position_number` registram a mesma largada; fica a de menos nulos (ou a combinacao das duas).
- **Pole:** `rd_race_pole_position` coincide com a classificacao em 1o em quase todas as linhas (tabela abaixo).
- **Voltas na classificacao:** `sess_quali_laps` depende de ate que fase do mata-mata (Q1/Q2/Q3) o piloto avancou. Por isso se relaciona com o alvo, mas como reflexo mais fraco da propria posicao de classificacao.
- **Tempo de grid:** `sess_grid_time_millis` e o tempo do ultimo segmento de classificacao disputado.
- **Pista:** comprimento e curvas aparecem em `race_*`, `layout_*` e `circuit_*`; o numero de voltas acompanha o comprimento e a distancia e quase fixa. Fica `race_course_length` e `race_turns`, que descrevem a configuracao usada em cada corrida.
- **Motor:** cilindrada e aspiracao mudam em degraus entre temporadas e so marcam a epoca, que `race_year` ja traz. `engine_configurations` nao mostrou associacao com o alvo no `04_Correlation.ipynb`.

In [9]:
def concordancia(a, b):
    ambas = df[a].notna() & df[b].notna()
    return df.loc[ambas, a].eq(df.loc[ambas, b]).mean()

print(f'Grid rd x sess iguais: {concordancia("rd_race_grid_position_number", "sess_grid_position_number"):.2%}')
display(df[['rd_race_grid_position_number', 'sess_grid_position_number']].isna().mean().mul(100).round(2).rename('nulos_%').to_frame())

display(pd.crosstab(df['rd_race_pole_position'], df['sess_quali_position_number'].eq(1).rename('quali_em_1o')))

display(df[['sess_quali_laps', 'sess_quali_position_number', ALVO]].corr(method='spearman').round(3))
faixa_quali = pd.cut(df['sess_quali_position_number'], [0, 10, 15, 20, 30]).rename('faixa_classificacao')
display(df.groupby(faixa_quali, observed=True)['sess_quali_laps'].median().rename('mediana_voltas').to_frame())

ultimo_segmento = (
    df['sess_quali_q3_millis']
    .combine_first(df['sess_quali_q2_millis'])
    .combine_first(df['sess_quali_q1_millis'])
    .combine_first(df['sess_quali_time_millis'])
)
com_tempo = df['sess_grid_time_millis'].notna() & ultimo_segmento.notna()
igual_tempo = df.loc[com_tempo, 'sess_grid_time_millis'].eq(ultimo_segmento[com_tempo]).mean()
print(f'sess_grid_time_millis = ultimo segmento da classificacao: {igual_tempo:.2%}')

pista = ['race_course_length', 'layout_length', 'circuit_length', 'race_turns', 'circuit_turns', 'race_laps', 'race_distance']
display(df[pista].corr(method='spearman').round(3))
print(df['race_distance'].describe(percentiles=[0.05, 0.95]).round(1).to_string())

epoca = df.groupby('race_year')[['engine_capacities', 'engine_aspirations']].agg(
    lambda serie: ', '.join(sorted(serie.dropna().astype(str).unique()))
)
display(epoca)

Grid rd x sess iguais: 99.93%


,nulos_%
rd_race_grid_position_number,2.13
sess_grid_position_number,1.55


quali_em_1o,False,True
rd_race_pole_position,,
0,8052,7
1,7,396


,sess_quali_laps,sess_quali_position_number,rd_position_display_order
sess_quali_laps,1.000,-0.550,-0.305
sess_quali_position_number,-0.550,1.000,0.627
rd_position_display_order,-0.305,0.627,1.000


,mediana_voltas
faixa_classificacao,
"(0, 10]",18.0
"(10, 15]",14.0
"(15, 20]",8.0
"(20, 30]",8.0


sess_grid_time_millis = ultimo segmento da classificacao: 99.96%


,race_course_length,layout_length,circuit_length,race_turns,circuit_turns,race_laps,race_distance
race_course_length,1.000,1.000,0.967,0.318,0.295,-0.997,0.458
layout_length,1.000,1.000,0.967,0.318,0.295,-0.997,0.457
circuit_length,0.967,0.967,1.000,0.284,0.318,-0.964,0.499
race_turns,0.318,0.318,0.284,1.000,0.920,-0.311,0.059
circuit_turns,0.295,0.295,0.318,0.920,1.000,-0.286,0.075
race_laps,-0.997,-0.997,-0.964,-0.311,-0.286,1.000,-0.416
race_distance,0.458,0.457,0.499,0.059,0.075,-0.416,1.000


count    8462.0
mean      304.7
std        10.8
min       253.8
5%        260.5
95%       309.7
max       310.4


,engine_capacities,engine_aspirations
race_year,,
2003,3.0,NATURALLY_ASPIRATED
2004,3.0,NATURALLY_ASPIRATED
2005,3.0,NATURALLY_ASPIRATED
2006,"2.4, 3.0",NATURALLY_ASPIRATED
2007,2.4,NATURALLY_ASPIRATED
2008,2.4,NATURALLY_ASPIRATED
2009,2.4,NATURALLY_ASPIRATED
2010,2.4,NATURALLY_ASPIRATED
2011,2.4,NATURALLY_ASPIRATED


## 9. Tempos brutos e sinal fraco

O tempo de volta bruto depende do tamanho da pista: acompanha o comprimento e quase nao se relaciona com o resultado. Dividido pelo melhor tempo da mesma sessao na mesma corrida, ele passa a medir o piloto. Por isso a coluna bruta e inutil, mas pode dar origem a uma feature relativa. Q2, Q3 e o tempo unico de 2003-2005 ainda tem muitos nulos por construcao.

As voltas dadas nos treinos livres quase nao se relacionam com o resultado. O fornecedor de pneus so varia nos primeiros anos da base; depois e unico para todos.

In [10]:
tempos = [
    'sess_fp1_time_millis', 'sess_fp2_time_millis', 'sess_fp3_time_millis',
    'sess_quali_time_millis', 'sess_quali_q1_millis', 'sess_quali_q2_millis', 'sess_quali_q3_millis',
]
relativo = df[tempos] / df.groupby('race_id')[tempos].transform('min')
display(pd.DataFrame({
    'nulos_%': df[tempos].isna().mean().mul(100).round(1),
    'bruto_x_comprimento': df[tempos].corrwith(df['race_course_length'], method='spearman'),
    'bruto_x_alvo': df[tempos].corrwith(df[ALVO], method='spearman'),
    'relativo_x_alvo': relativo.corrwith(df[ALVO], method='spearman'),
}).round(3))

voltas = ['sess_fp1_laps', 'sess_fp2_laps', 'sess_fp3_laps', 'sess_quali_laps']
display(df[voltas].corrwith(df[ALVO], method='spearman').round(3).rename('spearman_x_alvo').to_frame())

fornecedores = df.groupby('race_year')['tyremfr_id'].nunique()
print('Anos com mais de um fornecedor de pneus:', fornecedores[fornecedores.gt(1)].index.tolist())

,nulos_%,bruto_x_comprimento,bruto_x_alvo,relativo_x_alvo
sess_fp1_time_millis,9.9,0.782,0.072,0.479
sess_fp2_time_millis,3.3,0.792,0.074,0.507
sess_fp3_time_millis,5.2,0.760,0.074,0.504
sess_quali_time_millis,89.3,0.713,0.058,0.475
sess_quali_q1_millis,12.5,0.767,0.062,0.506
sess_quali_q2_millis,35.7,0.772,0.038,0.490
sess_quali_q3_millis,59.4,0.773,0.043,0.520


,spearman_x_alvo
sess_fp1_laps,-0.026
sess_fp2_laps,-0.019
sess_fp3_laps,0.060
sess_quali_laps,-0.305


Anos com mais de um fornecedor de pneus: [2003, 2004, 2005, 2006]


## 10. Constantes dentro da corrida

Pela secao 1 do `04_Correlation.ipynb`, o alvo e sempre uma permutacao de 1..N em cada corrida. Uma coluna com o mesmo valor para todos os pilotos da corrida nao ajuda a ordena-los: so se relaciona com o alvo por meio de N.

Elas nao sao inuteis em absoluto. Um modelo de arvore pode combina-las com outras colunas (por exemplo, equipe x pista de rua), e `race_year` e necessario para a divisao temporal. Mas sao as primeiras candidatas a sair de um modelo linear.

In [11]:
constantes = categorias['9. constante dentro da corrida']
valores_por_corrida = df.groupby('race_id')[constantes].nunique(dropna=False).max()
display(valores_por_corrida.rename('max_valores_por_corrida').to_frame())
assert valores_por_corrida.eq(1).all()

,max_valores_por_corrida
race_year,1
race_round,1
race_qualifying_format,1
race_direction,1
race_course_length,1
race_turns,1
circuit_id,1
circuit_type,1
layout_id,1
layout_effective,1


## 11. Tabela final

Uma linha por coluna, com a categoria, o percentual de nulos e a marcacao de `leakage_risk`.

In [12]:
tabela_final = (
    classificacao.to_frame()
    .join(df.isna().mean().mul(100).round(1).rename('nulos_%'))
    .join(risco)
    .rename_axis('coluna')
    .sort_values(['categoria', 'coluna'])
)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 80):
    display(tabela_final)

,categoria,nulos_%,leakage_risk
coluna,,,
rd_position_number,1. duplicata do alvo,18.5,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_position_text,1. duplicata do alvo,0.0,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_points,1. duplicata do alvo,55.1,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_driver_of_the_day,2. resultado da propria corrida,56.3,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_fastest_lap,2. resultado da propria corrida,0.0,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_grand_slam,2. resultado da propria corrida,0.0,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_laps,2. resultado da propria corrida,0.7,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_pit_stops,2. resultado da propria corrida,0.0,Campo de resultado/sessão sem liberação explícita como pré-largada
rd_race_positions_gained,2. resultado da propria corrida,18.5,Campo de resultado/sessão sem liberação explícita como pré-largada


## 12. Conclusoes

- **Vazamento (categorias 1 a 5):** duplicatas do alvo, estatisticas da propria corrida, standings, resumos de carreira atuais e agregados da temporada nao podem entrar no modelo. Os resumos de carreira sao os mais traicoeiros: parecem historico, mas sao fotografias de hoje.
- **Redundancia (categorias 6 e 7):** nomes, textos e medidas repetidas nao acrescentam informacao; de cada grupo fica uma coluna.
- **Forma bruta inutil (categoria 8):** os tempos so servem convertidos em diferenca para o melhor da sessao; voltas nos treinos livres e fornecedor de pneus tem sinal fraco.
- **Depende do modelo (categoria 9):** colunas constantes dentro da corrida so ajudam em interacoes.
- **O que sobra na forma bruta:** posicoes de grid, classificacao e FP1/FP2/FP3, e os ids de piloto, equipe e motor. O restante do sinal deve vir de features derivadas sem vazamento: idade na corrida, tempos relativos e os historicos `*_prior_*` de `07_Refining_Dataset.ipynb`.